In [ ]:
import re
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'SAE.py').exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
ANALYSIS_DIR = REPO_ROOT / 'llm' / 'analysis'
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

import SAE
from SAE_analysis_functions import *
import numpy as np
import torch
import matplotlib.pyplot as plt
import math
import ot
import plotly.express as px
from plotting_tools import *
# ─── Paths ───────────────────────────────────────────────────────────────────
DATA_DIR        = REPO_ROOT / Path('datasets/noised_luther/potentials/EleutherAI__pythia-410m-deduped/raw_potentials')
MODEL_BASE_PATH = REPO_ROOT / Path('datasets/noised_luther/SAE_params')

# ─── Data loading params ─────────────────────────────────────────────────────
N_PER_DIR     = 20
#N_PER_DIR     = 400

EXT           = '.pt'
POTENTIAL_DIM = 1130   # input dimension passed to sae_encode_potential_batch

# ─── SAE model configurations ────────────────────────────────────────────────
SAE_PARAMETERS = {
    'TOPKAE_10':     {'architecture': 'TopKAE',   'l1': '0',    'hidden_dim': 10,  'top_K': 2},
    'TOPKAE_40':     {'architecture': 'TopKAE',   'l1': '0',    'hidden_dim': 40,  'top_K': 4},
    'JUMPRELUAE_10': {'architecture': 'JumpReLU', 'l1': '1e-3', 'hidden_dim': 10,  'top_K': 0},
    'JUMPRELUAE_40': {'architecture': 'JumpReLU', 'l1': '1e-3', 'hidden_dim': 40,  'top_K': 0},
}

ARCH_REGISTRY = {
    'TopKAE':   TopKAE,
    'JumpReLU': JumpReLUAE,
    'GatedSAE': GatedSAE,
    'ReLUAE':   ReLUAE,
}


1. Load potential codes
2. Subset (optional)
3. Generate sublabels
4. Linear probe — confusion matrix & accuracy vs n
5. Linear probe — accuracy vs # PCs
6. Visualisation (PCA, UMAP, saturation)


### Load potential codes


In [ ]:
DIR_NAMES = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])
print(DIR_NAMES)

all_tensors = []
ns    = []   # int corruption intensity per sample
types = []   # corruption type string per sample
potential_code_dict = {}

for name in DIR_NAMES:
    m = re.match(r'^(\d+)_(.+)$', name)
    if m is None:
        raise ValueError(f'Unexpected directory name format: {name!r}')
    n_val, type_val = int(m.group(1)), m.group(2)
    d = DATA_DIR / name
    pt_files = sorted(d.rglob(f'*{EXT}'))
    if len(pt_files) < N_PER_DIR:
        raise ValueError(f'{name}: only {len(pt_files)} files, need {N_PER_DIR}')
    for p in pt_files[:N_PER_DIR]:
        all_tensors.append(torch.load(p)['object'])
    ns.extend([n_val]    * N_PER_DIR)
    types.extend([type_val] * N_PER_DIR)

stacked_tensor = torch.stack(
    [t if torch.is_tensor(t) else torch.tensor(t) for t in all_tensors], dim=0
)
print(f'Loaded {stacked_tensor.shape[0]} samples | '
      f'unique n: {sorted(set(ns))} | unique types: {sorted(set(types))}')

for key, cfg in SAE_PARAMETERS.items():
    print(key)
    model_path = MODEL_BASE_PATH / key / 'sparse_ae.pt'
    potential_codes = sae_encode_potential_batch(
        stacked_tensor, model_path, cfg['architecture'],
        POTENTIAL_DIM, cfg['hidden_dim'], cfg['top_K'],
      #  device='cuda',
        device='cpu',
        batch_size=1024,
    )
    potential_code_dict[key] = potential_codes

# Quick shape check
for key, codes in potential_code_dict.items():
    print(key, codes.shape)


### Subset (optional)


In [ ]:
def get_subset(code_dict, ns, types, n_range=None, type_filter=None):
    """
    Filter an entire code_dict plus its ns / types in one shot.

    Args:
        code_dict   : dict[str, Tensor]
        ns          : list[int]   — corruption intensity per sample
        types       : list[str]   — corruption type per sample
        n_range     : (min_n, max_n) inclusive, or None for all
        type_filter : list of type strings to keep, or None for all

    Returns:
        filtered_dict  : dict[str, Tensor]
        filtered_ns    : list[int]
        filtered_types : list[str]
    """
    mask = []
    for n, t in zip(ns, types):
        keep = True
        if n_range      is not None: keep = keep and (n_range[0] <= n <= n_range[1])
        if type_filter  is not None: keep = keep and (t in type_filter)
        mask.append(keep)

    idx            = torch.tensor(mask)
    filtered_ns    = [n for n, m in zip(ns,    mask) if m]
    filtered_types = [t for t, m in zip(types, mask) if m]
    filtered_dict  = {key: codes[idx] for key, codes in code_dict.items()}
    return filtered_dict, filtered_ns, filtered_types


subset_dict, subset_ns, subset_types = get_subset(
    potential_code_dict, ns, types,
    n_range=(0, 300),
    type_filter=['delete_words', 'permute_letters', 'global_swap_words', 'duplicate_words'],
)
for key, codes in subset_dict.items():
    print(f'{key}: {codes.shape}  |  unique n: {sorted(set(subset_ns))}  |  unique types: {sorted(set(subset_types))}')


In [ ]:
# ── Choose which dict to analyse — swap these two blocks to switch ───────────
ACTIVE_DICT  = potential_code_dict
ACTIVE_NS    = ns
ACTIVE_TYPES = types

# ACTIVE_DICT  = subset_dict
# ACTIVE_NS    = subset_ns
# ACTIVE_TYPES = subset_types

# Derived: combined label string (used by saturation plot & summarize helpers)
ACTIVE_LABELS = [f'{n}_{t}' for n, t in zip(ACTIVE_NS, ACTIVE_TYPES)]

print(f'Active: {len(ACTIVE_NS)} samples | '
      f'n values: {sorted(set(ACTIVE_NS))} | '
      f'types: {sorted(set(ACTIVE_TYPES))}')


### Generating sublabels


In [ ]:
# ACTIVE_TYPES is already stripped of n-prefixes
labels_no_intensities = ACTIVE_TYPES

# Collapsed variants (e.g. merges global/local swap) — still derived from ACTIVE_LABELS
labels_minimal       = summarize_corruption_labels(ACTIVE_LABELS, strip_numeric_prefix=True,  collapse_swap=True)
labels_collapse_swap = summarize_corruption_labels(ACTIVE_LABELS, strip_numeric_prefix=False, collapse_swap=True)

print('types (no_intensities):', sorted(set(labels_no_intensities)))
print('minimal:               ', sorted(set(labels_minimal)))


### Linear probe — confusion matrix & accuracy vs n


In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score,
                              confusion_matrix as sk_confusion_matrix)


def logistic_probe(
    Z, types, ns,
    test_size=0.2, n_splits=5, seed=0,
    standardize=True, C=1.0, max_iter=5000, class_weight='balanced',
):
    """
    Cross-validated logistic regression probing corruption-type classification.

    Returns a dict with:
        accuracy_mean    : float
        macro_f1_mean    : float
        confusion_matrix : (K, K) ndarray, row-normalised (recall per class),
                           summed across folds then normalised
        class_names      : list[str]
        acc_vs_n         : dict {n_val -> accuracy}  (OOS predictions, all folds pooled)
    """
    X      = Z.detach().float().cpu().numpy()
    ns_arr = np.asarray(ns, dtype=int)

    le          = LabelEncoder()
    y           = le.fit_transform(np.asarray(types))
    class_names = list(le.classes_)
    K           = len(class_names)

    steps = ([('scaler', StandardScaler())] if standardize else []) + [
        ('clf', LogisticRegression(
            C=C, solver='lbfgs', max_iter=max_iter, class_weight=class_weight))
    ]
    model    = Pipeline(steps)
    splitter = StratifiedShuffleSplit(n_splits=n_splits, test_size=test_size, random_state=seed)

    accs, f1s  = [], []
    cm_sum     = np.zeros((K, K), dtype=float)
    oos_true, oos_pred, oos_ns = [], [], []

    for i, (tr, te) in enumerate(splitter.split(X, y), 1):
        print(f'  split {i}/{n_splits}', end='\r', flush=True)
        model.fit(X[tr], y[tr])
        pred = model.predict(X[te])

        accs.append(accuracy_score(y[te], pred))
        f1s.append(f1_score(y[te], pred, average='macro'))
        cm_sum += sk_confusion_matrix(y[te], pred, labels=np.arange(K))

        oos_true.extend(y[te])
        oos_pred.extend(pred)
        oos_ns.extend(ns_arr[te])
    print()

    # Row-normalise: each row = recall for that true class
    row_sums = cm_sum.sum(axis=1, keepdims=True)
    cm_norm  = np.divide(cm_sum, row_sums, where=row_sums > 0, out=np.zeros_like(cm_sum))

    # Accuracy vs n from pooled OOS predictions
    oos_true = np.array(oos_true)
    oos_pred = np.array(oos_pred)
    oos_ns   = np.array(oos_ns)
    acc_vs_n = {
        int(n_val): float(accuracy_score(
            oos_true[oos_ns == n_val], oos_pred[oos_ns == n_val]
        ))
        for n_val in sorted(np.unique(oos_ns))
    }

    return {
        'accuracy_mean':    float(np.mean(accs)),
        'macro_f1_mean':    float(np.mean(f1s)),
        'confusion_matrix': cm_norm,
        'class_names':      class_names,
        'acc_vs_n':         acc_vs_n,
    }


def probe_all_models(code_dict, types, ns, **kwargs):
    """Run logistic_probe on every model in code_dict."""
    results = {}
    for key, codes in code_dict.items():
        print(key)
        results[key] = logistic_probe(codes, types, ns, **kwargs)
    return results


In [ ]:
def plot_confusion_matrices(probe_results, ncols=2, figsize_per_col=(4.5, 4.5)):
    """Heatmap of row-normalised confusion matrix for each model."""
    items  = list(probe_results.keys())
    n      = len(items)
    ncols  = min(ncols, n) if n > 0 else 1
    nrows  = math.ceil(n / ncols)
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize_per_col[0] * ncols, figsize_per_col[1] * nrows)
    )
    axes = np.atleast_1d(np.array(axes).ravel())

    for ax, key in zip(axes, items):
        cm  = probe_results[key]['confusion_matrix']
        cls = probe_results[key]['class_names']
        K   = len(cls)
        im  = ax.imshow(cm, vmin=0, vmax=1, cmap='Blues')
        ax.set_xticks(range(K))
        ax.set_yticks(range(K))
        ax.set_xticklabels(cls, rotation=35, ha='right', fontsize=7)
        ax.set_yticklabels(cls, fontsize=7)
        for i in range(K):
            for j in range(K):
                ax.text(j, i, f'{cm[i, j]:.2f}', ha='center', va='center',
                        fontsize=7, color='white' if cm[i, j] > 0.55 else 'black')
        ax.set_title(key, fontsize=9)
        ax.set_xlabel('Predicted', fontsize=8)
        ax.set_ylabel('True', fontsize=8)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    for ax in axes[n:]:
        ax.axis('off')
    fig.suptitle('Confusion matrices (row-normalised recall)', y=1.01)
    fig.tight_layout()
    plt.show()


def plot_acc_vs_n(probe_results, figsize=(8, 4)):
    """All models on one axes: accuracy vs corruption intensity n."""
    fig, ax = plt.subplots(figsize=figsize)
    for key, res in probe_results.items():
        d  = res['acc_vs_n']
        xs = sorted(d.keys())
        ax.plot(xs, [d[x] for x in xs], marker='o', label=key)
    ax.set_xlabel('Corruption intensity')
    ax.set_ylabel('Accuracy')
    ax.set_title('Classification accuracy vs corruption intensity')
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()


In [ ]:
probe_results = probe_all_models(ACTIVE_DICT, ACTIVE_TYPES, ACTIVE_NS)

# Summary table
print(f'\n{"Model":<20} {"Accuracy":>10} {"Macro F1":>10}')
print('-' * 42)
for key, res in probe_results.items():
    print(f'{key:<20} {res["accuracy_mean"]:>10.3f} {res["macro_f1_mean"]:>10.3f}')

plot_confusion_matrices(probe_results, ncols=2)
plot_acc_vs_n(probe_results)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.preprocessing import LabelEncoder


def class_atom_mass_matrix(
    Z,
    labels,
    *,
    normalize='by_atom',   # 'by_atom', 'by_class', or 'none'
    eps=1e-12,
    sort_atoms=True,
    sort_by='purity',       # 'purity', 'argmax', 'mass', or None
):
    """
    Build a class-by-atom mass matrix from nonnegative codes.

    Parameters
    ----------
    Z : torch.Tensor or ndarray, shape (N, K)
        Nonnegative codes.
    labels : array-like, shape (N,)
        Class labels.
    normalize :
        'by_atom'  -> each column sums to 1:
                      M[c,j] = class-c share of atom-j total mass
        'by_class' -> each row sums to 1:
                      M[c,j] = atom-j share of class-c total mass
        'none'     -> raw class-conditional mass sums
    sort_atoms :
        Whether to reorder atoms for cleaner visualization.
    sort_by :
        'purity' -> sort by max class share (most class-specific first)
        'argmax' -> group by dominant class, then purity
        'mass'   -> sort by total atom mass
        None     -> keep original order
    """
    X = Z.detach().float().cpu().numpy() if torch.is_tensor(Z) else np.asarray(Z, dtype=float)
    y = np.asarray(labels)

    if X.ndim != 2:
        raise ValueError(f"Z must be 2D, got shape {X.shape}")
    if len(y) != X.shape[0]:
        raise ValueError(f"len(labels)={len(y)} but Z has {X.shape[0]} rows")
    if np.any(X < -1e-10):
        raise ValueError("Z appears to contain negative entries; this routine assumes nonnegative codes")

    X = np.clip(X, 0.0, None)

    le = LabelEncoder()
    y_enc = le.fit_transform(y)
    class_names = list(le.classes_)
    C = len(class_names)
    N, K = X.shape

    # raw class-by-atom mass
    M_raw = np.zeros((C, K), dtype=float)
    for c in range(C):
        M_raw[c] = X[y_enc == c].sum(axis=0)

    if normalize == 'by_atom':
        denom = M_raw.sum(axis=0, keepdims=True)
        M = np.divide(M_raw, denom + eps)
    elif normalize == 'by_class':
        denom = M_raw.sum(axis=1, keepdims=True)
        M = np.divide(M_raw, denom + eps)
    elif normalize == 'none':
        M = M_raw.copy()
    else:
        raise ValueError("normalize must be one of {'by_atom','by_class','none'}")

    order = np.arange(K)

    if sort_atoms:
        if normalize == 'by_atom':
            purity = M.max(axis=0)
            dominant = M.argmax(axis=0)
        else:
            # use by-atom shares just for sorting, even if plotting another normalization
            M_atom = np.divide(M_raw, M_raw.sum(axis=0, keepdims=True) + eps)
            purity = M_atom.max(axis=0)
            dominant = M_atom.argmax(axis=0)

        total_mass = M_raw.sum(axis=0)

        if sort_by == 'purity':
            order = np.lexsort((-total_mass, -purity))
            order = order[::-1]
        elif sort_by == 'argmax':
            order = np.lexsort((-total_mass, -purity, dominant))
        elif sort_by == 'mass':
            order = np.argsort(-total_mass)
        elif sort_by is None:
            order = np.arange(K)
        else:
            raise ValueError("sort_by must be one of {'purity','argmax','mass', None}")

        M = M[:, order]
        M_raw = M_raw[:, order]

    return {
        'matrix': M,
        'matrix_raw': M_raw,
        'class_names': class_names,
        'atom_order': order,
    }


def plot_class_atom_matrix(
    result,
    *,
    title=None,
    figsize=None,
    cmap='viridis',
    show_values=False,
    value_fmt='{:.2f}',
    vmax=None,
):
    M = result['matrix']
    class_names = result['class_names']
    C, K = M.shape

    if figsize is None:
        figsize = (max(8, 0.28 * K), max(2.5, 0.7 * C))

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(M, aspect='auto', interpolation='nearest', cmap=cmap, vmax=vmax)

    ax.set_yticks(np.arange(C))
    ax.set_yticklabels(class_names)
    ax.set_xticks(np.arange(K))
    ax.set_xticklabels(np.arange(K), rotation=90)
    ax.set_xlabel('Atom index (possibly reordered)')
    ax.set_ylabel('Class')

    if title is not None:
        ax.set_title(title)

    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label('Relative mass')

    if show_values and K <= 40:
        for r in range(C):
            for c in range(K):
                ax.text(c, r, value_fmt.format(M[r, c]),
                        ha='center', va='center', fontsize=8, color='white')

    plt.tight_layout()
    return fig, ax

In [ ]:
CLASS_ATOM_KEY = 'JUMPRELUAE_10'  # alternatives include 'JUMPRELUAE_40'

if CLASS_ATOM_KEY not in ACTIVE_DICT:
    raise KeyError(f'{CLASS_ATOM_KEY} not found. Available keys: {sorted(ACTIVE_DICT)}')

res = class_atom_mass_matrix(
    ACTIVE_DICT[CLASS_ATOM_KEY],
    ACTIVE_TYPES,
    normalize='by_atom',
    sort_atoms=True,
    sort_by='argmax',
)

plot_class_atom_matrix(
    res,
    figsize=(10, 4),
    title=f'Class-by-atom mass share ({CLASS_ATOM_KEY})',
)
plt.show()


### Linear probe — accuracy vs # PCs


In [ ]:
from sklearn.decomposition import PCA


def logistic_regression_pca_curve(
    Z, y, pca_dims,
    test_size=0.2, n_splits=10, seed=0, standardize=True,
    C=1.0, max_iter=5000, class_weight='balanced',
):
    if torch.is_tensor(y):
        y_np = y.detach().cpu().numpy().reshape(-1)
    else:
        y_np = np.asarray(y).reshape(-1)

    X = Z.detach().float().cpu().numpy()
    if len(y_np) != X.shape[0]:
        raise ValueError(f'len(y)={len(y_np)} but Z has N={X.shape[0]} rows')

    d        = X.shape[1]
    pca_dims = sorted(set(int(r) for r in pca_dims if 1 <= r <= d))
    if not pca_dims:
        raise ValueError(f'pca_dims must contain ints in [1, {d}]')

    splitter  = StratifiedShuffleSplit(n_splits=n_splits, test_size=test_size, random_state=seed)
    accs_by_r = {r: [] for r in pca_dims}
    f1s_by_r  = {r: [] for r in pca_dims}

    for i, (tr, te) in enumerate(splitter.split(X, y_np), 1):
        print(f'  split {i}/{n_splits}', end='\r', flush=True)
        Xtr, Xte = X[tr], X[te]
        ytr, yte = y_np[tr], y_np[te]

        if standardize:
            scaler = StandardScaler()
            Xtr    = scaler.fit_transform(Xtr)
            Xte    = scaler.transform(Xte)

        pca    = PCA(n_components=max(pca_dims), random_state=seed)
        Xtr_p  = pca.fit_transform(Xtr)
        Xte_p  = pca.transform(Xte)

        for r in pca_dims:
            clf  = LogisticRegression(C=C, solver='lbfgs', max_iter=max_iter, class_weight=class_weight)
            clf.fit(Xtr_p[:, :r], ytr)
            pred = clf.predict(Xte_p[:, :r])
            accs_by_r[r].append(accuracy_score(yte, pred))
            f1s_by_r[r].append(f1_score(yte, pred, average='macro'))
    print()

    return {r: {'accuracy_mean': float(np.mean(accs_by_r[r])),
                'macro_f1_mean': float(np.mean(f1s_by_r[r]))} for r in pca_dims}


def run_pca_curve(code_dict, types, pca_dims):
    """Run PCA-dim vs accuracy for each model's codes."""
    all_results = {}
    for key, codes in code_dict.items():
        print(key)
        res = logistic_regression_pca_curve(codes, types, pca_dims)
        all_results[key] = {r: res[r]['accuracy_mean'] for r in res}
    return all_results


def plot_pca_curves_grid(all_results, ncols=3, figsize_per_colrow=(5, 3.5)):
    items = list(all_results.keys())
    n     = len(items)
    ncols = min(ncols, n) if n > 0 else 1
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(
        nrows, ncols,
        figsize=(figsize_per_colrow[0] * ncols, figsize_per_colrow[1] * nrows),
        sharex=True, sharey=True
    )
    axes = np.atleast_1d(np.array(axes).ravel())
    for ax, item in zip(axes, items):
        d    = all_results[item]
        dims = sorted(d.keys())
        ax.plot(dims, [d[r] for r in dims], marker='o')
        ax.set_title(str(item))
        ax.set_xlabel('# PCs')
        ax.set_ylabel('Accuracy (mean)')
        ax.grid(True, alpha=0.3)
    for ax in axes[n:]:
        ax.axis('off')
    fig.suptitle('Linear classifier accuracy vs # principal components', y=1.02)
    fig.tight_layout()
    plt.show()


pca_dims   = [1, 2, 3, 4, 5, 6, 7, 8]
pca_results = run_pca_curve(ACTIVE_DICT, ACTIVE_TYPES, pca_dims)
plot_pca_curves_grid(pca_results, ncols=2)


### Wasserstein distance vs PC1 of potential codes

In [ ]:
from scipy.stats import pearsonr, spearmanr

# ─── Paths for activation data ───────────────────────────────────────────────
NOISED_ACT_DIR = REPO_ROOT / Path('datasets/noised_luther/activations/noised_activations/EleutherAI__pythia-410m-deduped')
BASE_ACT_PATH  = REPO_ROOT / Path('datasets/noised_luther/activations/base/luther_L2_residual.pt')

# ─── Subset fraction ─────────────────────────────────────────────────────────
W2_SUBSET_PCT = 5   # percent of data to use; increase for a less noisy estimate
k_pct = W2_SUBSET_PCT
seed   = 42

# ─── 1. Load noised activations in the SAME order as potentials ──────────────
# Rebuild the exact file list used when loading potentials so indices match.
potential_file_list = []
for name in DIR_NAMES:
    d = DATA_DIR / name
    pt_files = sorted(d.rglob(f'*{EXT}'))[:N_PER_DIR]
    potential_file_list.extend(pt_files)

# Load noised activations by matching each potential file's *name* inside the
# corresponding subdirectory of NOISED_ACT_DIR.
noised_acts_list = []
for pf in potential_file_list:
    subdir = pf.parent.name                       # e.g. '100_delete_words'
    fname  = pf.name                               # e.g. 'seed_164_L2_residual.pt'
    act_path = NOISED_ACT_DIR / subdir / fname
    if not act_path.exists():
        raise FileNotFoundError(
            f'Expected activation file not found: {act_path}\n'
            f'(matching potential file {pf})')
    noised_acts_list.append(torch.load(act_path)['acts'])

print(f'Loaded {len(noised_acts_list)} noised activation point-clouds')
print(f'Example shape: {noised_acts_list[0].shape}')

# ─── 2. Load base activations ────────────────────────────────────────────────
base_acts = torch.load(BASE_ACT_PATH)['acts']
print(f'Base activation shape: {base_acts.shape}')

# ─── 3. Take a k% random subset ─────────────────────────────────────────────
N_total    = len(noised_acts_list)
n_subset   = max(1, int(N_total * k_pct / 100))
rng        = np.random.default_rng(seed)
subset_idx = np.sort(rng.choice(N_total, size=n_subset, replace=False))
print(f'Using {n_subset}/{N_total} samples ({k_pct}%)')

# ─── 4. Compute 2-Wasserstein distances (POT) ───────────────────────────────
base_np = base_acts.float().cpu().numpy()
# Uniform weights for the base point-cloud (shared across all comparisons)
a_weights = np.ones(base_np.shape[0]) / base_np.shape[0]

w2_distances = np.empty(n_subset, dtype=np.float64)
for i, idx in enumerate(subset_idx):
    target_np = noised_acts_list[idx].float().cpu().numpy()
    b_weights = np.ones(target_np.shape[0]) / target_np.shape[0]
    # Squared-Euclidean cost matrix
    M = ot.dist(base_np, target_np, metric='sqeuclidean')
    # EMD with squared cost → take sqrt for W2
    w2_distances[i] = np.sqrt(ot.emd2(a_weights, b_weights, M))
    if (i + 1) % max(1, n_subset // 10) == 0:
        print(f'  computed {i+1}/{n_subset}')

print(f'W2 distances — min: {w2_distances.min():.4f}, '
      f'max: {w2_distances.max():.4f}, mean: {w2_distances.mean():.4f}')

# ─── 5. PC1 of potential codes on the same subset ────────────────────────────
W2_PC1_RESULTS = {}
for key, codes in potential_code_dict.items():
    codes_subset = codes[subset_idx].float().cpu().numpy()
    scaler = StandardScaler()
    codes_std = scaler.fit_transform(codes_subset)
    pca = PCA(n_components=1, random_state=0)
    pc1 = pca.fit_transform(codes_std).ravel()

    r_pearson, p_pearson   = pearsonr(w2_distances, pc1)
    r_spearman, p_spearman = spearmanr(w2_distances, pc1)

    W2_PC1_RESULTS[key] = {
        'pc1': pc1,
        'pearson_r': r_pearson,  'pearson_p': p_pearson,
        'spearman_r': r_spearman, 'spearman_p': p_spearman,
        'explained_var': float(pca.explained_variance_ratio_[0]),
    }

    print(f'\n{key}:')
    print(f'  PC1 explained variance: {pca.explained_variance_ratio_[0]:.3f}')
    print(f'  Pearson  r={r_pearson:.4f}  (p={p_pearson:.2e})')
    print(f'  Spearman r={r_spearman:.4f}  (p={p_spearman:.2e})')

# ─── 6. Scatter plots ────────────────────────────────────────────────────────
n_models = len(W2_PC1_RESULTS)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4), squeeze=False)
axes = axes.ravel()

subset_types_w2 = [ACTIVE_TYPES[i] for i in subset_idx]
unique_types = sorted(set(subset_types_w2))
cmap = plt.cm.tab10

for ax, (key, res) in zip(axes, W2_PC1_RESULTS.items()):
    for j, t in enumerate(unique_types):
        mask = np.array([s == t for s in subset_types_w2])
        ax.scatter(w2_distances[mask], res['pc1'][mask],
                   s=12, alpha=0.5, label=t, color=cmap(j))
    ax.set_xlabel('W2 distance (noised vs base)')
    ax.set_ylabel('PC1 of potential codes')
    ax.set_title(f'{key}\nr={res["pearson_r"]:.3f} (p={res["pearson_p"]:.1e})')
    ax.legend(fontsize=7, loc='best')
    ax.grid(True, alpha=0.3)

fig.suptitle(f'W2 distance vs PC1 of potential codes ({k_pct}% subset, n={n_subset})', y=1.02)
fig.tight_layout()
plt.show()

### Visualisation


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score


def diagnose_pc1(
    Z, y, *, standardize=True, bins=40, permute_seed=0, title='PC1 diagnostic',
):
    X     = Z.detach().float().cpu().numpy() if torch.is_tensor(Z) else np.asarray(Z, dtype=np.float32)
    y_arr = y.detach().cpu().numpy().reshape(-1) if torch.is_tensor(y) else np.asarray(y).reshape(-1)
    if X.shape[0] != len(y_arr):
        raise ValueError(f'Z has N={X.shape[0]} rows but y has len={len(y_arr)}')
    if X.shape[0] == 0:
        raise ValueError('diagnose_pc1 got an empty input. Check the n_range/type_filter used to build subset_dict.')

    le      = LabelEncoder()
    y_int   = le.fit_transform(y_arr)
    classes = le.classes_
    K       = len(classes)

    Xp  = StandardScaler().fit_transform(X) if standardize else X
    pca = PCA(n_components=1, random_state=0)
    z   = pca.fit_transform(Xp).reshape(-1)
    plt.figure(figsize=(8, 4))
    for k, cls in enumerate(classes):
        plt.hist(z[y_int == k], bins=bins, density=True, alpha=0.35, label=str(cls))
    plt.title(title)
    plt.xlabel('PC1 coordinate (z)')
    plt.ylabel('Density')
    plt.legend(loc='best', fontsize=8)
    #plt.xlim(-4.1, 4.1)
    plt.tight_layout()
    plt.show()

    metrics = {'explained_variance_ratio_pc1': float(pca.explained_variance_ratio_[0]),
               'num_classes': int(K)}

    if K == 2:
        auc      = roc_auc_score(y_int, z)
        ts       = np.quantile(z, np.linspace(0.01, 0.99, 199))
        best_acc = max(accuracy_score(y_int, (z >= t).astype(int)) for t in ts)
        metrics.update({'binary_auc_pc1': float(auc), 'binary_best_thresh_acc_pc1': float(best_acc)})
    else:
        clf1d = LogisticRegression(solver='lbfgs', max_iter=2000)
        clf1d.fit(z.reshape(-1, 1), y_int)
        probs = clf1d.predict_proba(z.reshape(-1, 1))
        metrics['multiclass_ovr_auc_pc1_macro'] = float(
            roc_auc_score(y_int, probs, multi_class='ovr', average='macro'))

    rng    = np.random.default_rng(permute_seed)
    y_perm = rng.permutation(y_int)
    if K == 2:
        metrics['binary_auc_pc1_shuffled'] = float(roc_auc_score(y_perm, z))
    else:
        clf1d.fit(z.reshape(-1, 1), y_perm)
        probs_perm = clf1d.predict_proba(z.reshape(-1, 1))
        metrics['multiclass_ovr_auc_pc1_macro_shuffled'] = float(
            roc_auc_score(y_perm, probs_perm, multi_class='ovr', average='macro'))

    return metrics




In [ ]:
subset_dict, subset_ns, subset_types = get_subset(
    potential_code_dict, ns, types,
    n_range=(1, 25),
    type_filter=['delete_words', 'permute_letters', 'global_swap_words', 'duplicate_words'],
)
for key, codes in subset_dict.items():
    print(f'{key}: {codes.shape}  |  unique n: {sorted(set(subset_ns))}  |  unique types: {sorted(set(subset_types))}')


In [ ]:

# ── Set which model key to visualise in the cells below ──────────────────────
VISUALIZE_KEY  = 'TOPKAE_40'
potential_codes = subset_dict[VISUALIZE_KEY]
types=subset_types

diagnose_pc1(
    potential_codes, types,
    standardize=True, bins=40, permute_seed=0,
    title=f'First component of potential codes (TopK40, low corruptions)',
)